# Ablation 4: Hard Negatives Mining (3 Strategies)

**Research Question**: Open: does training with DDI hard negatives (no-wound images) reduce false positives.

**Strategies**:
- **4a**: Mixed training (wounds + DDI from scratch)
- **4b**: Best checkpoint fine-tuning on DDI only
- **4c**: Last checkpoint fine-tuning on DDI only


In [ ]:
# Lock NumPy to prevent ultralytics from upgrading to 2.x
import numpy as np
import subprocess

default_numpy = np.__version__
print(f"Installing ultralytics...")
subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)

print(f"Locking NumPy to {default_numpy}...")
subprocess.run(["pip", "install", "-q", "--force-reinstall", f"numpy=={default_numpy}"], check=True)

import numpy as np
print(f"OK NumPy locked: {np.__version__}")


In [ ]:
import sys
from pathlib import Path
import yaml
import shutil
import pandas as pd

# Environment
IN_KAGGLE = Path("/kaggle/input").exists()
BASE_INPUT_PATH = Path("/kaggle/input") if IN_KAGGLE else Path.cwd()
OUTPUT_PATH = Path("/kaggle/working") if IN_KAGGLE else Path("/tmp/ablation_4")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Code dataset
CODE_PATH = BASE_INPUT_PATH / "wound-seg-code-v1"
if CODE_PATH.exists():
    sys.path.insert(0, str(CODE_PATH))

print("OK Environment configured")


In [ ]:
# Strategy 4a: Copy wound dataset + add DDI hard negatives
print("=" * 80)
print("CREATING MIXED DATASET (WOUNDS + DDI)")
print("=" * 80)

# Copy baseline wound dataset
WOUND_SRC = BASE_INPUT_PATH / "baseline-outputs" / "data_wound_seg"
MIXED_DATASET = OUTPUT_PATH / "mixed_dataset"

print("Copying wound YOLO dataset...")
shutil.copytree(WOUND_SRC, MIXED_DATASET)

# Load DDI dataset and create train/test split
DDI_METADATA_PATH = BASE_INPUT_PATH / "ddi-diverse-dermatology"
DDI_IMAGES_PATH = BASE_INPUT_PATH / "dermatology-data"
ddi_metadata = DDI_METADATA_PATH / "ddi_metadata_cleaned.csv"
ddi_df = pd.read_csv(ddi_metadata)

# 80/20 train/test split (stratified by malignant status)
from sklearn.model_selection import train_test_split
ddi_train_df, ddi_test_df = train_test_split(
    ddi_df,
    test_size=0.2,
    random_state=42,
    stratify=ddi_df['malignant']
)
print(f"DDI split: {len(ddi_train_df)} train, {len(ddi_test_df)} test")

# DDI images are directly in the root directory
ddi_images_src = DDI_IMAGES_PATH
png_count = len(list(ddi_images_src.glob("*.png")))
print(f"\nOK DDI image directory: {ddi_images_src}")
print(f"  Total images available: {png_count}")
assert png_count == 642, f"Expected 642 PNG images, found {png_count} in {ddi_images_src}"

# Copy DDI images to mixed dataset train
train_images_dst = MIXED_DATASET / "train" / "images"
train_labels_dst = MIXED_DATASET / "train" / "labels"

print(f"\nAdding {len(ddi_train_df)} DDI hard negatives to training set...")
ddi_copied = 0
for idx, row in ddi_train_df.iterrows():
    img_filename = row['DDI_file']
    src_img = ddi_images_src / img_filename

    if src_img.exists():
        # Copy image with ddi_ prefix
        dst_img = train_images_dst / f"ddi_{img_filename}"
        shutil.copy2(src_img, dst_img)

        # Create empty label (no wound = empty file)
        label_file = train_labels_dst / f"ddi_{src_img.stem}.txt"
        label_file.touch()

        ddi_copied += 1

print(f"OK Added {ddi_copied} DDI hard negatives")

# Verify mixed dataset
train_imgs = len(list((MIXED_DATASET / "train" / "images").glob("*.png")))
train_lbls = len(list((MIXED_DATASET / "train" / "labels").glob("*.txt")))
test_imgs = len(list((MIXED_DATASET / "test" / "images").glob("*.png")))
test_lbls = len(list((MIXED_DATASET / "test" / "labels").glob("*.txt")))

print(f"\nOK Mixed Dataset:")
print(f"  Train: {train_imgs} images, {train_lbls} labels (2208 wound + {ddi_copied} DDI)")
print(f"  Test: {test_imgs} images, {test_lbls} labels")

# Create config for 4a
mixed_config_path = OUTPUT_PATH / "wound_dataset_mixed.yaml"
with open(mixed_config_path, 'w') as f:
    f.write(f"""path: {MIXED_DATASET}
train: train
val: test

names:
  0: wound
""")

print(f"\nOK Mixed config: {mixed_config_path}")


In [ ]:
# Strategies 4b/4c: DDI-only dataset for fine-tuning
print("=" * 80)
print("CREATING DDI-ONLY DATASET")
print("=" * 80)

DDI_ONLY_DATASET = OUTPUT_PATH / "ddi_only_dataset"
(DDI_ONLY_DATASET / "train" / "images").mkdir(parents=True, exist_ok=True)
(DDI_ONLY_DATASET / "train" / "labels").mkdir(parents=True, exist_ok=True)
(DDI_ONLY_DATASET / "test" / "images").mkdir(parents=True, exist_ok=True)
(DDI_ONLY_DATASET / "test" / "labels").mkdir(parents=True, exist_ok=True)

# Copy DDI train
ddi_train_copied = 0
for idx, row in ddi_train_df.iterrows():
    img_filename = row['DDI_file']
    src_img = ddi_images_src / img_filename

    if src_img.exists():
        dst_img = DDI_ONLY_DATASET / "train" / "images" / img_filename
        shutil.copy2(src_img, dst_img)

        # Empty label
        label_file = DDI_ONLY_DATASET / "train" / "labels" / f"{src_img.stem}.txt"
        label_file.touch()
        ddi_train_copied += 1

# Copy DDI test (using split from above)
ddi_test_copied = 0
for idx, row in ddi_test_df.iterrows():
    img_filename = row['DDI_file']
    src_img = ddi_images_src / img_filename

    if src_img.exists():
        dst_img = DDI_ONLY_DATASET / "test" / "images" / img_filename
        shutil.copy2(src_img, dst_img)

        # Empty label
        label_file = DDI_ONLY_DATASET / "test" / "labels" / f"{src_img.stem}.txt"
        label_file.touch()
        ddi_test_copied += 1

print(f"OK DDI-only dataset:")
print(f"  Train: {ddi_train_copied} images")
print(f"  Test: {ddi_test_copied} images")

# Create config for 4b/4c
ddi_config_path = OUTPUT_PATH / "ddi_only_dataset.yaml"
with open(ddi_config_path, 'w') as f:
    f.write(f"""path: {DDI_ONLY_DATASET}
train: train
val: test

names:
  0: wound
""")

print(f"OK DDI-only config: {ddi_config_path}")


In [ ]:
from ultralytics import YOLO
import json
import time

RANDOM_SEEDS = [42, 123, 456]
all_results = {}

# Strategy 4a: Mixed training (from scratch)
print("\n" + "=" * 80)
print("STRATEGY 4a: MIXED TRAINING (WOUNDS + DDI)")
print("=" * 80)

for seed in RANDOM_SEEDS:
    print(f"\nSeed {seed}...")
    model = YOLO("yolov8n-seg.pt")

    start = time.time()
    results = model.train(
        data=str(mixed_config_path),
        epochs=100,
        batch=32,
        lr0=0.015,
        workers=8,
        amp=False,
        imgsz=512,
        patience=10,
        seed=seed,
        project=str(OUTPUT_PATH / "yolo_training_4a"),
        name=f"seed_{seed}",
        exist_ok=True,
        verbose=False
    )

    duration = (time.time() - start) / 60
    best_path = OUTPUT_PATH / "yolo_training_4a" / f"seed_{seed}" / "weights" / "best.pt"
    last_path = OUTPUT_PATH / "yolo_training_4a" / f"seed_{seed}" / "weights" / "last.pt"

    all_results[f"4a_seed_{seed}"] = {
        "strategy": "4a_mixed",
        "seed": seed,
        "duration_min": round(duration, 1),
        "best_model": str(best_path),
        "last_model": str(last_path)
    }

    print(f"OK 4a seed {seed} complete in {duration:.1f} min")

print("\nOK Strategy 4a complete")

# Strategy 4b: Best checkpoint fine-tuning on DDI
print("\n" + "=" * 80)
print("STRATEGY 4b: BEST CHECKPOINT FINE-TUNING ON DDI")
print("=" * 80)

for seed in RANDOM_SEEDS:
    print(f"\nSeed {seed}...")
    best_4a = OUTPUT_PATH / "yolo_training_4a" / f"seed_{seed}" / "weights" / "best.pt"
    model = YOLO(str(best_4a))

    start = time.time()
    results = model.train(
        data=str(ddi_config_path),
        epochs=20,  # Shorter fine-tuning
        batch=32,
        lr0=0.001,  # Lower LR for fine-tuning
        workers=8,
        amp=False,
        imgsz=512,
        patience=5,
        seed=seed,
        project=str(OUTPUT_PATH / "yolo_training_4b"),
        name=f"seed_{seed}",
        exist_ok=True,
        verbose=False
    )

    duration = (time.time() - start) / 60
    all_results[f"4b_seed_{seed}"] = {
        "strategy": "4b_best_finetune",
        "seed": seed,
        "duration_min": round(duration, 1),
        "base_model": str(best_4a)
    }

    print(f"OK 4b seed {seed} complete in {duration:.1f} min")

print("\nOK Strategy 4b complete")

# Strategy 4c: Last checkpoint fine-tuning on DDI
print("\n" + "=" * 80)
print("STRATEGY 4c: LAST CHECKPOINT FINE-TUNING ON DDI")
print("=" * 80)

for seed in RANDOM_SEEDS:
    print(f"\nSeed {seed}...")
    last_4a = OUTPUT_PATH / "yolo_training_4a" / f"seed_{seed}" / "weights" / "last.pt"
    model = YOLO(str(last_4a))

    start = time.time()
    results = model.train(
        data=str(ddi_config_path),
        epochs=20,
        batch=32,
        lr0=0.001,
        workers=8,
        amp=False,
        imgsz=512,
        patience=5,
        seed=seed,
        project=str(OUTPUT_PATH / "yolo_training_4c"),
        name=f"seed_{seed}",
        exist_ok=True,
        verbose=False
    )

    duration = (time.time() - start) / 60
    all_results[f"4c_seed_{seed}"] = {
        "strategy": "4c_last_finetune",
        "seed": seed,
        "duration_min": round(duration, 1),
        "base_model": str(last_4a)
    }

    print(f"OK 4c seed {seed} complete in {duration:.1f} min")

print("\nOK Strategy 4c complete")
print("\n" + "=" * 80)
print("ALL 3 STRATEGIES COMPLETE")
print("=" * 80)


In [ ]:
# Save completion status
status = {
    "ablation": "4_hard_negatives",
    "status": "complete",
    "strategies": ["4a_mixed", "4b_best_finetune", "4c_last_finetune"],
    "seeds": RANDOM_SEEDS,
    "results": all_results
}

with open(OUTPUT_PATH / "completion_status.json", 'w') as f:
    json.dump(status, f, indent=2)

print("OK Ablation 4 complete - all 3 strategies trained!")
